# Demo — Vecinos semánticos frente a vecinos léxicos

> Demo que acompaña a [Modelos de
> embeddings](../bloque1/01c-embeddings.qmd). Los embeddings se
> calcularon una vez con `paraphrase-multilingual-MiniLM-L12-v2` (384
> dimensiones) y se guardaron como datos estáticos. Por eso la demo no
> necesita ninguna API ni descargar un modelo.

Elige una frase y mira cuáles son sus vecinas más cercanas en el espacio
de embeddings. La columna **coseno** usa los vectores del modelo. La
columna **palabras en común** cuenta solo coincidencias de vocabulario
(sin palabras vacías como *el* o *de*).

Busca casos en los que el coseno sea alto y las palabras en común sean
pocas: ahí está la diferencia entre similitud semántica y léxica. Puedes
pulsar cualquier punto del mapa (o una fila de la tabla) para usar esa
frase como consulta.

``` shinylive-python
#| '!! shinylive warning !!': |
#|   shinylive does not work in self-contained HTML documents.
#|   Please set `embed-resources: false` in your metadata.
#| standalone: true
#| viewerHeight: 1180

## file: app.py
import html
import re

import numpy as np
from shiny import reactive
from shiny.express import input, ui, render

from data import DATA

sentences = DATA["sentences"]
topics = DATA["topics"]
xy = np.array(DATA["xy"])
cos = np.array(DATA["cosine"])

TOPIC_COLORS = {"ia": "#306998", "cocina": "#e65100", "finanzas": "#2e7d32", "deporte": "#c99a00"}
TOPIC_NAMES = {"ia": "IA", "cocina": "Cocina", "finanzas": "Finanzas", "deporte": "Deporte"}
STOP = {"el", "la", "los", "las", "de", "del", "que", "y", "a", "en", "un", "una", "por",
        "para", "con", "se", "su", "sus", "al", "es", "lo", "o", "ya", "si", "sobre"}
MODES = {"sem": "Significado (coseno de embeddings)", "lex": "Palabras en común (léxico)"}

ui.tags.style("""
body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif}
.card{border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;margin:10px 0;background:#fff}
.muted{color:#6b7280;font-size:.85em}
.pt{cursor:pointer}.pt:hover circle{stroke:#111;stroke-width:2}
.nb{width:100%;font-size:.9em;border-collapse:collapse}
.nb td,.nb th{padding:5px 6px;border-bottom:1px solid #f3f4f6;vertical-align:middle}
.nb th{font-weight:600;font-size:.85em;text-align:left;color:#374151}
.nb tr{cursor:pointer}.nb tbody tr:hover{background:#f9fafb}
.track{display:inline-block;width:70px;height:10px;background:#f3f4f6;border-radius:3px;vertical-align:middle;margin-right:6px}
.fill{display:block;height:100%;background:#306998;border-radius:3px}
mark{background:#FFD43B;padding:0 2px;border-radius:2px}
.zero{color:#e65100;font-weight:600}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px;vertical-align:middle}
.score{font-size:1.05em;font-weight:600}
""")


def words(text):
    return {w for w in re.findall(r"\w+", text.lower()) if w not in STOP}


def shared(a, b):
    return words(a) & words(b)


def highlight(text, keep):
    out = []
    for piece in re.split(r"(\w+)", text):
        out.append(f"<mark>{html.escape(piece)}</mark>" if piece.lower() in keep else html.escape(piece))
    return "".join(out)


@reactive.effect
@reactive.event(input.pick)
def _pick():
    ui.update_select("query", selected=str(int(input.pick())))


@reactive.calc
def neighbors():
    q = int(input.query())
    others = [j for j in range(len(sentences)) if j != q]
    if input.mode() == "sem":
        order = sorted(others, key=lambda j: -cos[q, j])
    else:
        def lex(j):
            a, b = words(sentences[q]), words(sentences[j])
            return (len(a & b), len(a & b) / (len(a | b) or 1))
        # Los empates (muy frecuentes, casi todo a 0 palabras) se deshacen con un orden aleatorio fijo,
        # no por índice: las frases están agrupadas por tema y el índice regalaría vecinas del mismo tema.
        tie = np.random.default_rng(q).permutation(len(sentences))
        order = sorted(others, key=lambda j: (-lex(j)[0], -lex(j)[1], tie[j]))
    return q, order[: input.k()]


ui.input_select(
    "query", "Frase de consulta (o pulsa un punto del mapa)",
    {str(i): s for i, s in enumerate(sentences)}, selected="0", width="100%",
)
with ui.div(style="display:flex;gap:24px;flex-wrap:wrap;align-items:flex-end"):
    ui.input_radio_buttons("mode", "Buscar vecinas por", MODES, selected="sem", inline=True)
    ui.input_slider("k", "Vecinas a mostrar", 1, 6, 3)


@render.ui
def scatter():
    q, nb = neighbors()
    lex = input.mode() == "lex"
    W, H, pad = 640, 380, 28
    lo, hi = xy.min(axis=0), xy.max(axis=0)
    px = lambda i: pad + (xy[i, 0] - lo[0]) / (hi[0] - lo[0]) * (W - 2 * pad)
    py = lambda i: H - pad - (xy[i, 1] - lo[1]) / (hi[1] - lo[1]) * (H - 2 * pad)
    line_col = "#e65100" if lex else "#306998"
    dash = "stroke-dasharray='6 4'" if lex else ""
    lines = "".join(
        f"<line x1='{px(q):.1f}' y1='{py(q):.1f}' x2='{px(j):.1f}' y2='{py(j):.1f}' stroke='{line_col}' "
        f"stroke-width='{1 + 5 * max(cos[q, j], 0):.1f}' stroke-opacity='.55' "
        f"{dash}/>"
        for j in nb
    )
    pts = []
    for i, s in enumerate(sentences):
        col = TOPIC_COLORS[topics[i]]
        r = 11 if i == q else 7
        ring = (
            f"<circle cx='{px(i):.1f}' cy='{py(i):.1f}' r='16' fill='none' stroke='#111' stroke-width='2'/>"
            if i == q else ""
        )
        rank = ""
        if i in nb:
            k = nb.index(i) + 1
            rank = (
                f"<text x='{px(i) + 10:.1f}' y='{py(i) - 8:.1f}' font-size='12' font-weight='700' fill='#111' "
                f"stroke='#fff' stroke-width='3' paint-order='stroke'>{k}</text>"
            )
        pts.append(
            f"<g class='pt' onclick=\"Shiny.setInputValue('pick', {i}, {{priority: 'event'}})\">"
            f"<title>{html.escape(s)}</title>{ring}"
            f"<circle cx='{px(i):.1f}' cy='{py(i):.1f}' r='{r}' fill='{col}' fill-opacity='.85' stroke='#fff' stroke-width='1.5'/>"
            f"{rank}</g>"
        )
    legend = " · ".join(
        f"<span class='dot' style='background:{c}'></span>{TOPIC_NAMES[t]}" for t, c in TOPIC_COLORS.items()
    )
    return ui.HTML(
        f"<div class='card'><b>Mapa 2D de los embeddings</b> <span class='muted'>(PCA de 384 dimensiones a 2)</span>"
        f"<svg viewBox='0 0 {W} {H}' width='100%' style='display:block;margin-top:4px' "
        f"font-family='system-ui,sans-serif'>{lines}{''.join(pts)}</svg>"
        f"<div class='muted'>{legend} · círculo negro = consulta · números = vecinas · "
        f"pasa el ratón por un punto para leer la frase y púlsalo para usarla como consulta.</div></div>"
    )


@render.ui
def table():
    q, nb = neighbors()
    lex = input.mode() == "lex"
    same = sum(topics[j] == topics[q] for j in nb)
    all_shared = set().union(*(shared(sentences[q], sentences[j]) for j in nb)) if nb else set()
    rows = []
    for rank, j in enumerate(nb, start=1):
        common = shared(sentences[q], sentences[j])
        c = max(cos[q, j], 0)
        n_common = f"<span class='zero'>0</span>" if not common else str(len(common))
        rows.append(
            f"<tr onclick=\"Shiny.setInputValue('pick', {j}, {{priority: 'event'}})\" title='Usar como consulta'>"
            f"<td>{rank}</td><td>{highlight(sentences[j], common)}</td>"
            f"<td style='white-space:nowrap'><span class='dot' style='background:{TOPIC_COLORS[topics[j]]}'></span>{TOPIC_NAMES[topics[j]]}</td>"
            f"<td style='white-space:nowrap'><span class='track'><span class='fill' style='width:{100 * c:.0f}%'></span></span>"
            f"{cos[q, j]:.2f}</td><td style='text-align:center'>{n_common}</td></tr>"
        )
    color = "#2e7d32" if same == len(nb) else ("#e65100" if same * 2 >= len(nb) else "#c62828")
    criterion = "por palabras en común" if lex else "por significado"
    tip = (
        "Con pocas palabras compartidas, el orden léxico es casi arbitrario: empates a 0 palabras."
        if lex else "Fíjate en las vecinas con <span class='zero'>0</span> palabras en común: "
        "el embedding las encuentra igualmente."
    )
    return ui.HTML(
        f"<div class='card'><p style='margin:0 0 6px'><b>Consulta:</b> {highlight(sentences[q], all_shared)}</p>"
        f"<p class='score' style='color:{color};margin:0 0 6px'>{same} de {len(nb)} vecinas "
        f"{criterion} son del mismo tema ({TOPIC_NAMES[topics[q]]}).</p>"
        f"<table class='nb'><thead><tr><th>#</th><th>Vecina</th><th>Tema</th><th>Coseno</th>"
        f"<th>Palabras en común</th></tr></thead><tbody>{''.join(rows)}</tbody></table>"
        f"<p class='muted' style='margin:6px 0 0'>{tip} En amarillo, las palabras compartidas con la consulta.</p></div>"
    )

## file: data.py
# DATA-START
DATA = {
 "model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
 "dim": 384,
 "sentences": [
  "El modelo de lenguaje genera texto coherente a partir de una instrucción.",
  "Las redes neuronales aprenden representaciones a partir de grandes cantidades de datos.",
  "El sistema responde preguntas consultando documentos almacenados.",
  "Entrenar un algoritmo requiere datos etiquetados de buena calidad.",
  "La inteligencia artificial ya escribe código y resume informes.",
  "Los transformers procesan secuencias prestando atención a todas sus partes.",
  "Un chatbot puede mantener una conversación sobre temas técnicos.",
  "El ordenador aprendió a reconocer rostros en las fotografías.",
  "Hay que dejar reposar la masa del pan durante una hora.",
  "Añade una pizca de sal y remueve la salsa a fuego lento.",
  "Este bizcocho queda esponjoso si montas bien las claras.",
  "La paella se hace mejor con arroz bomba y caldo casero.",
  "Asar las verduras con aceite de oliva realza su sabor.",
  "Para una tortilla jugosa, cocina los huevos a fuego suave.",
  "La inflación reduce el poder adquisitivo de los salarios.",
  "El banco central subió los tipos de interés para frenar los precios.",
  "Diversificar la cartera disminuye el riesgo de perder dinero.",
  "Las acciones de la empresa cayeron tras presentar pérdidas trimestrales.",
  "Ahorrar una parte fija del sueldo cada mes es una buena costumbre.",
  "El préstamo hipotecario se paga en cuotas durante veinte años.",
  "El delantero marcó dos goles en la segunda parte.",
  "El equipo necesita ganar el partido del domingo para clasificarse.",
  "Correr cinco kilómetros cada mañana mejora la resistencia.",
  "El portero detuvo el penalti en el último minuto.",
  "La selección entrenó bajo la lluvia antes del torneo.",
  "Nadar es un ejercicio completo que cuida las articulaciones."
 ],
 "topics": [
  "ia",
  "ia",
  "ia",
  "ia",
  "ia",
  "ia",
  "ia",
  "ia",
  "cocina",
  "cocina",
  "cocina",
  "cocina",
  "cocina",
  "cocina",
  "finanzas",
  "finanzas",
  "finanzas",
  "finanzas",
  "finanzas",
  "finanzas",
  "deporte",
  "deporte",
  "deporte",
  "deporte",
  "deporte",
  "deporte"
 ],
 "xy": [
  [
   0.8853999972343445,
   0.3474999964237213
  ],
  [
   0.9799000024795532,
   0.49219998717308044
  ],
  [
   0.9243000149726868,
   0.6496000289916992
  ],
  [
   0.9352999925613403,
   0.424699991941452
  ],
  [
   1.0,
   0.44279998540878296
  ],
  [
   0.935699999332428,
   0.44780001044273376
  ],
  [
   0.9567000269889832,
   0.49619999527931213
  ],
  [
   0.9307000041007996,
   0.3781999945640564
  ],
  [
   0.06809999793767929,
   0.23469999432563782
  ],
  [
   0.0,
   0.000699999975040555
  ],
  [
   0.22190000116825104,
   0.16300000250339508
  ],
  [
   0.17520000040531158,
   0.09459999948740005
  ],
  [
   0.13099999725818634,
   0.07840000092983246
  ],
  [
   0.10320000350475311,
   0.0
  ],
  [
   0.05889999866485596,
   1.0
  ],
  [
   0.14419999718666077,
   0.7986000180244446
  ],
  [
   0.1379999965429306,
   0.9984999895095825
  ],
  [
   0.16689999401569366,
   0.9157999753952026
  ],
  [
   0.3021000027656555,
   0.6736999750137329
  ],
  [
   0.26350000500679016,
   0.8668000102043152
  ],
  [
   0.15150000154972076,
   0.39899998903274536
  ],
  [
   0.35690000653266907,
   0.3449999988079071
  ],
  [
   0.38580000400543213,
   0.33959999680519104
  ],
  [
   0.06449999660253525,
   0.6568999886512756
  ],
  [
   0.31349998712539673,
   0.3495999872684479
  ],
  [
   0.5081999897956848,
   0.3926999866962433
  ]
 ],
 "cosine": [
  [
   1.0,
   0.358599990606308,
   0.2637999951839447,
   0.36070001125335693,
   0.45879998803138733,
   0.25699999928474426,
   0.22290000319480896,
   0.17630000412464142,
   0.08269999921321869,
   0.04149999842047691,
   0.05810000002384186,
   0.17180000245571136,
   0.066600002348423,
   0.09030000120401382,
   -0.09700000286102295,
   -0.04340000078082085,
   -0.034299999475479126,
   -0.14090000092983246,
   0.024299999698996544,
   -0.04410000145435333,
   -0.07190000265836716,
   0.026100000366568565,
   0.0835999995470047,
   -0.04859999939799309,
   0.09099999815225601,
   0.1363999992609024
  ],
  [
   0.358599990606308,
   1.0,
   0.27649998664855957,
   0.34869998693466187,
   0.4124999940395355,
   0.4359999895095825,
   0.26910001039505005,
   0.3813000023365021,
   -0.09260000288486481,
   -0.04619999974966049,
   0.01889999955892563,
   0.08879999816417694,
   -0.028200000524520874,
   0.012299999594688416,
   -0.0364999994635582,
   0.1306000053882599,
   0.03739999979734421,
   -0.09629999846220016,
   0.05640000104904175,
   -0.03180000185966492,
   0.03880000114440918,
   0.11670000106096268,
   0.16979999840259552,
   -0.09520000219345093,
   0.09809999912977219,
   0.09920000284910202
  ],
  [
   0.2637999951839447,
   0.27649998664855957,
   1.0,
   0.2833999991416931,
   0.36559998989105225,
   0.2198999971151352,
   0.3736000061035156,
   0.23800000548362732,
   -0.03660000115633011,
   -0.11230000108480453,
   0.04839999973773956,
   -0.061400000005960464,
   -0.06400000303983688,
   -0.10220000147819519,
   -0.004800000227987766,
   -0.03889999911189079,
   0.09910000115633011,
   0.031199999153614044,
   0.04729999974370003,
   0.061799999326467514,
   -0.07490000128746033,
   0.004399999976158142,
   -0.03959999978542328,
   -0.050700001418590546,
   -0.004000000189989805,
   0.012799999676644802
  ],
  [
   0.36070001125335693,
   0.34869998693466187,
   0.2833999991416931,
   1.0,
   0.5282999873161316,
   0.2824999988079071,
   0.30559998750686646,
   0.2865000069141388,
   -0.014100000262260437,
   -0.006599999964237213,
   -0.026599999517202377,
   0.2272000014781952,
   0.10700000077486038,
   0.06270000338554382,
   -0.00430000014603138,
   -0.13019999861717224,
   0.0340999998152256,
   -0.045099999755620956,
   0.14970000088214874,
   0.02590000070631504,
   -0.05050000175833702,
   0.14630000293254852,
   0.20649999380111694,
   -0.08959999680519104,
   0.07090000063180923,
   0.10300000011920929
  ],
  [
   0.45879998803138733,
   0.4124999940395355,
   0.36559998989105225,
   0.5282999873161316,
   1.0,
   0.40470001101493835,
   0.44449999928474426,
   0.36039999127388,
   0.0803999975323677,
   0.021900000050663948,
   0.07540000230073929,
   0.13910000026226044,
   0.034299999475479126,
   0.1378999948501587,
   -9.999999747378752e-05,
   -0.08020000159740448,
   0.01850000023841858,
   0.059300001710653305,
   0.0851999968290329,
   0.026900000870227814,
   0.008700000122189522,
   0.11159999668598175,
   0.0917000025510788,
   -0.008299999870359898,
   0.07349999994039536,
   0.06340000033378601
  ],
  [
   0.25699999928474426,
   0.4359999895095825,
   0.2198999971151352,
   0.2824999988079071,
   0.40470001101493835,
   1.0,
   0.2777000069618225,
   0.3774999976158142,
   -0.018699999898672104,
   0.032499998807907104,
   0.1518000066280365,
   -0.029600000008940697,
   -0.05849999934434891,
   0.07020000368356705,
   -0.07349999994039536,
   0.0820000022649765,
   -0.056699998676776886,
   -0.05429999902844429,
   0.030300000682473183,
   -0.03500000014901161,
   -0.027499999850988388,
   0.05719999969005585,
   0.1111999973654747,
   -0.03660000115633011,
   0.012199999764561653,
   0.1242000013589859
  ],
  [
   0.22290000319480896,
   0.26910001039505005,
   0.3736000061035156,
   0.30559998750686646,
   0.44449999928474426,
   0.2777000069618225,
   1.0,
   0.3208000063896179,
   0.0697999969124794,
   -0.004999999888241291,
   0.12039999663829803,
   -0.03709999844431877,
   -0.10360000282526016,
   -0.041999999433755875,
   -0.11640000343322754,
   -0.20589999854564667,
   0.056699998676776886,
   -0.014299999922513962,
   0.03819999843835831,
   -0.042100001126527786,
   -0.0940999984741211,
   0.08330000191926956,
   0.04740000143647194,
   0.06780000030994415,
   0.0034000000450760126,
   0.09889999777078629
  ],
  [
   0.17630000412464142,
   0.3813000023365021,
   0.23800000548362732,
   0.2865000069141388,
   0.36039999127388,
   0.3774999976158142,
   0.3208000063896179,
   1.0,
   -0.012600000016391277,
   -0.014999999664723873,
   0.1738000065088272,
   0.03799999877810478,
   0.04639999940991402,
   -0.0010999999940395355,
   -0.08079999685287476,
   -0.08900000154972076,
   -0.10339999943971634,
   -0.05640000104904175,
   -0.018799999728798866,
   -0.14429999887943268,
   0.06679999828338623,
   0.05939999967813492,
   0.03420000150799751,
   -0.06210000067949295,
   0.0835999995470047,
   0.0032999999821186066
  ],
  [
   0.08269999921321869,
   -0.09260000288486481,
   -0.03660000115633011,
   -0.014100000262260437,
   0.0803999975323677,
   -0.018699999898672104,
   0.0697999969124794,
   -0.012600000016391277,
   1.0,
   0.39500001072883606,
   0.4011000096797943,
   0.227400004863739,
   0.23389999568462372,
   0.3783000111579895,
   0.007899999618530273,
   -0.024399999529123306,
   0.07919999957084656,
   -0.00039999998989515007,
   0.1949000060558319,
   0.11259999871253967,
   0.07779999822378159,
   0.1543000042438507,
   0.22310000658035278,
   0.2433999925851822,
   0.0982000008225441,
   0.031300000846385956
  ],
  [
   0.04149999842047691,
   -0.04619999974966049,
   -0.11230000108480453,
   -0.006599999964237213,
   0.021900000050663948,
   0.032499998807907104,
   -0.004999999888241291,
   -0.014999999664723873,
   0.39500001072883606,
   1.0,
   0.4320000112056732,
   0.4778999984264374,
   0.42809998989105225,
   0.6477000117301941,
   -0.03999999910593033,
   0.032099999487400055,
   0.010700000450015068,
   -0.06800000369548798,
   0.04989999905228615,
   -0.08380000293254852,
   0.15410000085830688,
   0.08780000358819962,
   0.14650000631809235,
   0.12549999356269836,
   0.11010000109672546,
   0.007899999618530273
  ],
  [
   0.05810000002384186,
   0.01889999955892563,
   0.04839999973773956,
   -0.026599999517202377,
   0.07540000230073929,
   0.1518000066280365,
   0.12039999663829803,
   0.1738000065088272,
   0.4011000096797943,
   0.4320000112056732,
   1.0,
   0.22840000689029694,
   0.25529998540878296,
   0.33709999918937683,
   -0.021800000220537186,
   -0.026000000536441803,
   0.08579999953508377,
   -0.018200000748038292,
   0.02419999986886978,
   -0.10289999842643738,
   0.07599999755620956,
   0.15029999613761902,
   0.07000000029802322,
   0.13369999825954437,
   0.1624000072479248,
   0.0502999983727932
  ],
  [
   0.17180000245571136,
   0.08879999816417694,
   -0.061400000005960464,
   0.2272000014781952,
   0.13910000026226044,
   -0.029600000008940697,
   -0.03709999844431877,
   0.03799999877810478,
   0.227400004863739,
   0.4778999984264374,
   0.22840000689029694,
   1.0,
   0.44020000100135803,
   0.4876999855041504,
   0.008500000461935997,
   0.03849999979138374,
   0.011599999852478504,
   -0.035599999129772186,
   0.09619999676942825,
   -0.007300000172108412,
   0.17309999465942383,
   0.10869999974966049,
   0.1446000039577484,
   0.02879999950528145,
   0.09920000284910202,
   0.06880000233650208
  ],
  [
   0.066600002348423,
   -0.028200000524520874,
   -0.06400000303983688,
   0.10700000077486038,
   0.034299999475479126,
   -0.05849999934434891,
   -0.10360000282526016,
   0.04639999940991402,
   0.23389999568462372,
   0.42809998989105225,
   0.25529998540878296,
   0.44020000100135803,
   1.0,
   0.384799987077713,
   0.032099999487400055,
   -0.01899999938905239,
   -0.014700000174343586,
   -0.055799998342990875,
   0.09860000014305115,
   -0.18610000610351562,
   0.10199999809265137,
   0.031099999323487282,
   0.12999999523162842,
   -0.08879999816417694,
   0.08320000022649765,
   0.06840000301599503
  ],
  [
   0.09030000120401382,
   0.012299999594688416,
   -0.10220000147819519,
   0.06270000338554382,
   0.1378999948501587,
   0.07020000368356705,
   -0.041999999433755875,
   -0.0010999999940395355,
   0.3783000111579895,
   0.6477000117301941,
   0.33709999918937683,
   0.4876999855041504,
   0.384799987077713,
   1.0,
   -0.01759999990463257,
   0.014800000004470348,
   -0.05939999967813492,
   -0.08410000056028366,
   0.005400000140070915,
   -0.004399999976158142,
   0.09369999915361404,
   0.09470000118017197,
   0.10700000077486038,
   -0.019899999722838402,
   0.10589999705553055,
   0.021800000220537186
  ],
  [
   -0.09700000286102295,
   -0.0364999994635582,
   -0.004800000227987766,
   -0.00430000014603138,
   -9.999999747378752e-05,
   -0.07349999994039536,
   -0.11640000343322754,
   -0.08079999685287476,
   0.007899999618530273,
   -0.03999999910593033,
   -0.021800000220537186,
   0.008500000461935997,
   0.032099999487400055,
   -0.01759999990463257,
   1.0,
   0.32350000739097595,
   0.5928999781608582,
   0.39430001378059387,
   0.24130000174045563,
   0.15970000624656677,
   -0.024900000542402267,
   -0.06129999831318855,
   0.054099999368190765,
   0.1890999972820282,
   0.033900000154972076,
   -0.12160000205039978
  ],
  [
   -0.04340000078082085,
   0.1306000053882599,
   -0.03889999911189079,
   -0.13019999861717224,
   -0.08020000159740448,
   0.0820000022649765,
   -0.20589999854564667,
   -0.08900000154972076,
   -0.024399999529123306,
   0.032099999487400055,
   -0.026000000536441803,
   0.03849999979138374,
   -0.01899999938905239,
   0.014800000004470348,
   0.32350000739097595,
   1.0,
   0.25130000710487366,
   0.05889999866485596,
   0.03830000013113022,
   0.1720999926328659,
   0.027699999511241913,
   -0.04360000044107437,
   0.0007999999797903001,
   0.15330000221729279,
   0.023800000548362732,
   -0.05400000140070915
  ],
  [
   -0.034299999475479126,
   0.03739999979734421,
   0.09910000115633011,
   0.0340999998152256,
   0.01850000023841858,
   -0.056699998676776886,
   0.056699998676776886,
   -0.10339999943971634,
   0.07919999957084656,
   0.010700000450015068,
   0.08579999953508377,
   0.011599999852478504,
   -0.014700000174343586,
   -0.05939999967813492,
   0.5928999781608582,
   0.25130000710487366,
   1.0,
   0.351500004529953,
   0.2883000075817108,
   0.24580000340938568,
   -0.0031999999191612005,
   -0.045499999076128006,
   0.019500000402331352,
   0.20239999890327454,
   -0.07540000230073929,
   -0.04019999876618385
  ],
  [
   -0.14090000092983246,
   -0.09629999846220016,
   0.031199999153614044,
   -0.045099999755620956,
   0.059300001710653305,
   -0.05429999902844429,
   -0.014299999922513962,
   -0.05640000104904175,
   -0.00039999998989515007,
   -0.06800000369548798,
   -0.018200000748038292,
   -0.035599999129772186,
   -0.055799998342990875,
   -0.08410000056028366,
   0.39430001378059387,
   0.05889999866485596,
   0.351500004529953,
   1.0,
   0.035100001841783524,
   0.10620000213384628,
   0.0658000037074089,
   0.0012000000569969416,
   -0.08990000188350677,
   0.21559999883174896,
   0.08290000259876251,
   -0.01679999940097332
  ],
  [
   0.024299999698996544,
   0.05640000104904175,
   0.04729999974370003,
   0.14970000088214874,
   0.0851999968290329,
   0.030300000682473183,
   0.03819999843835831,
   -0.018799999728798866,
   0.1949000060558319,
   0.04989999905228615,
   0.02419999986886978,
   0.09619999676942825,
   0.09860000014305115,
   0.005400000140070915,
   0.24130000174045563,
   0.03830000013113022,
   0.2883000075817108,
   0.035100001841783524,
   1.0,
   0.2264000028371811,
   -0.023499999195337296,
   0.000699999975040555,
   0.24660000205039978,
   -0.0008999999845400453,
   0.04780000075697899,
   0.08020000159740448
  ],
  [
   -0.04410000145435333,
   -0.03180000185966492,
   0.061799999326467514,
   0.02590000070631504,
   0.026900000870227814,
   -0.03500000014901161,
   -0.042100001126527786,
   -0.14429999887943268,
   0.11259999871253967,
   -0.08380000293254852,
   -0.10289999842643738,
   -0.007300000172108412,
   -0.18610000610351562,
   -0.004399999976158142,
   0.15970000624656677,
   0.1720999926328659,
   0.24580000340938568,
   0.10620000213384628,
   0.2264000028371811,
   1.0,
   0.0071000000461936,
   -0.03959999978542328,
   -0.0714000016450882,
   0.05260000005364418,
   -0.16359999775886536,
   -0.02290000021457672
  ],
  [
   -0.07190000265836716,
   0.03880000114440918,
   -0.07490000128746033,
   -0.05050000175833702,
   0.008700000122189522,
   -0.027499999850988388,
   -0.0940999984741211,
   0.06679999828338623,
   0.07779999822378159,
   0.15410000085830688,
   0.07599999755620956,
   0.17309999465942383,
   0.10199999809265137,
   0.09369999915361404,
   -0.024900000542402267,
   0.027699999511241913,
   -0.0031999999191612005,
   0.0658000037074089,
   -0.023499999195337296,
   0.0071000000461936,
   1.0,
   0.34610000252723694,
   0.1348000019788742,
   0.32679998874664307,
   0.19349999725818634,
   0.06300000101327896
  ],
  [
   0.026100000366568565,
   0.11670000106096268,
   0.004399999976158142,
   0.14630000293254852,
   0.11159999668598175,
   0.05719999969005585,
   0.08330000191926956,
   0.05939999967813492,
   0.1543000042438507,
   0.08780000358819962,
   0.15029999613761902,
   0.10869999974966049,
   0.031099999323487282,
   0.09470000118017197,
   -0.06129999831318855,
   -0.04360000044107437,
   -0.045499999076128006,
   0.0012000000569969416,
   0.000699999975040555,
   -0.03959999978542328,
   0.34610000252723694,
   1.0,
   0.1817999929189682,
   0.23960000276565552,
   0.41769999265670776,
   0.07699999958276749
  ],
  [
   0.0835999995470047,
   0.16979999840259552,
   -0.03959999978542328,
   0.20649999380111694,
   0.0917000025510788,
   0.1111999973654747,
   0.04740000143647194,
   0.03420000150799751,
   0.22310000658035278,
   0.14650000631809235,
   0.07000000029802322,
   0.1446000039577484,
   0.12999999523162842,
   0.10700000077486038,
   0.054099999368190765,
   0.0007999999797903001,
   0.019500000402331352,
   -0.08990000188350677,
   0.24660000205039978,
   -0.0714000016450882,
   0.1348000019788742,
   0.1817999929189682,
   1.0,
   0.01600000075995922,
   0.2924000024795532,
   0.27730000019073486
  ],
  [
   -0.04859999939799309,
   -0.09520000219345093,
   -0.050700001418590546,
   -0.08959999680519104,
   -0.008299999870359898,
   -0.03660000115633011,
   0.06780000030994415,
   -0.06210000067949295,
   0.2433999925851822,
   0.12549999356269836,
   0.13369999825954437,
   0.02879999950528145,
   -0.08879999816417694,
   -0.019899999722838402,
   0.1890999972820282,
   0.15330000221729279,
   0.20239999890327454,
   0.21559999883174896,
   -0.0008999999845400453,
   0.05260000005364418,
   0.32679998874664307,
   0.23960000276565552,
   0.01600000075995922,
   1.0,
   0.23919999599456787,
   -0.012900000438094139
  ],
  [
   0.09099999815225601,
   0.09809999912977219,
   -0.004000000189989805,
   0.07090000063180923,
   0.07349999994039536,
   0.012199999764561653,
   0.0034000000450760126,
   0.0835999995470047,
   0.0982000008225441,
   0.11010000109672546,
   0.1624000072479248,
   0.09920000284910202,
   0.08320000022649765,
   0.10589999705553055,
   0.033900000154972076,
   0.023800000548362732,
   -0.07540000230073929,
   0.08290000259876251,
   0.04780000075697899,
   -0.16359999775886536,
   0.19349999725818634,
   0.41769999265670776,
   0.2924000024795532,
   0.23919999599456787,
   1.0,
   0.18940000236034393
  ],
  [
   0.1363999992609024,
   0.09920000284910202,
   0.012799999676644802,
   0.10300000011920929,
   0.06340000033378601,
   0.1242000013589859,
   0.09889999777078629,
   0.0032999999821186066,
   0.031300000846385956,
   0.007899999618530273,
   0.0502999983727932,
   0.06880000233650208,
   0.06840000301599503,
   0.021800000220537186,
   -0.12160000205039978,
   -0.05400000140070915,
   -0.04019999876618385,
   -0.01679999940097332,
   0.08020000159740448,
   -0.02290000021457672,
   0.06300000101327896,
   0.07699999958276749,
   0.27730000019073486,
   -0.012900000438094139,
   0.18940000236034393,
   1.0
  ]
 ]
}
# DATA-END
```

## Qué observar

- Las frases de un mismo tema se agrupan, aunque no compartan palabras.
  El modelo entiende que *“Nadar cuida las articulaciones”* y *“Correr
  mejora la resistencia”* pertenecen al mismo campo.
- Un método léxico, como contar palabras, falla con sinónimos y
  paráfrasis. Un embedding no depende de que las palabras coincidan.
- Cambia **«Buscar vecinas por»** a *Palabras en común* y compara el
  marcador «x de k vecinas son del mismo tema». Casi todas las frases
  comparten 0 o 1 palabras con la consulta, así que el orden léxico es
  poco más que un empate.
- La proyección 2D es una simplificación: de 384 dimensiones a dos se
  pierde información. Las distancias entre grupos en el dibujo pueden
  ser engañosas, pero los vecinos cercanos en el espacio original casi
  siempre se ven cerca.

> **Para probar en clase**
>
> Cambia el modelo por otro (por ejemplo `all-MiniLM-L6-v2` o
> `e5-small-v2`) en `scripts/build_demo_data.py` y compara qué vecinos
> cambian.